# Tutorial 2: Kriging (Hard Data Only)

**Corresponds to MATLAB `BMEHRLIBtutorial.m`**

This notebook demonstrates:
1. Simple / Ordinary kriging on a 2-D grid
2. Effect of covariance model choice on RMSE
3. Kriging variance map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pybme import bme_predict, build_cov_matrix

## 1. Generate Synthetic Data

We create a known sinusoidal field and sample 25 noisy "hard" measurements.

In [ ]:
rng = np.random.default_rng(12)

def true_field(x, y):
    return 50 + 20 * np.sin(x / 3000) + 10 * np.cos(y / 2000)

# Hard data (simulating soil sampling)
n_hard = 25
ch = np.column_stack([
    rng.uniform(178000, 195000, n_hard),
    rng.uniform(90000, 108000, n_hard),
])
zh = true_field(ch[:, 0], ch[:, 1]) + rng.normal(0, 5.0, n_hard)

# Estimation grid (17×19)
gx, gy = np.meshgrid(
    np.linspace(178000, 194000, 17),
    np.linspace(90000, 108000, 19),
)
ck = np.column_stack([gx.ravel(), gy.ravel()])
z_ref = true_field(ck[:, 0], ck[:, 1])

print(f"Hard data:       {n_hard} points")
print(f"Estimation grid: {ck.shape[0]} points (17×19)")

## 2. Compare Covariance Models

Run ordinary kriging with four different covariance specifications and compare RMSE.

In [ ]:
cov_configs = [
    ("Exponential",    "exponential", [100.0, 5000.0]),
    ("Gaussian (RBF)", "gaussian",    [100.0, 5000.0]),
    ("Spherical",      "spherical",   [100.0, 8000.0]),
    ("Nested nug+exp", ["nugget", "exponential"], [[10.0], [90.0, 5000.0]]),
]

print(f"{'Model':20s}  {'RMSE':>8s}  {'Mean var':>10s}")
print("-" * 44)
for name, model, params in cov_configs:
    results = bme_predict(ck, ch, zh, model=model, params=params,
                          nhmax=10, dmax=15000.0, order=0)
    z_est = np.array([r.mean for r in results])
    v_est = np.array([r.variance for r in results])
    rmse = np.sqrt(np.mean((z_est - z_ref) ** 2))
    print(f"{name:20s}  {rmse:8.3f}  {v_est.mean():10.2f}")

## 3. Simple Kriging vs Ordinary Kriging

- **Simple Kriging** assumes a known, fixed mean (`order=NaN`, `mean_prior=50`)
- **Ordinary Kriging** estimates a local constant mean (`order=0`)

In [ ]:
# Simple Kriging
results_sk = bme_predict(ck, ch, zh, model="exponential",
                         params=[100.0, 5000.0],
                         nhmax=10, dmax=15000.0,
                         order=float("nan"), mean_prior=50.0)
z_sk = np.array([r.mean for r in results_sk])
rmse_sk = np.sqrt(np.mean((z_sk - z_ref) ** 2))

# Ordinary Kriging
results_ok = bme_predict(ck, ch, zh, model="exponential",
                         params=[100.0, 5000.0],
                         nhmax=10, dmax=15000.0, order=0)
z_ok = np.array([r.mean for r in results_ok])
v_ok = np.array([r.variance for r in results_ok])
rmse_ok = np.sqrt(np.mean((z_ok - z_ref) ** 2))

print(f"Simple Kriging   RMSE = {rmse_sk:.3f}")
print(f"Ordinary Kriging RMSE = {rmse_ok:.3f}")

## 4. Visualization: True Field, Kriging Estimate, and Kriging Variance

In [ ]:
nx, ny = 17, 19

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

# True field
im0 = axes[0].pcolormesh(gx, gy, z_ref.reshape(ny, nx),
                          cmap='hot', shading='auto')
axes[0].scatter(ch[:, 0], ch[:, 1], c='cyan', marker='v', s=20, zorder=5)
axes[0].set_title('True Field')
plt.colorbar(im0, ax=axes[0])

# Ordinary kriging estimate
im1 = axes[1].pcolormesh(gx, gy, z_ok.reshape(ny, nx),
                          cmap='hot', shading='auto')
axes[1].scatter(ch[:, 0], ch[:, 1], c='cyan', marker='v', s=20, zorder=5)
axes[1].set_title(f'Ordinary Kriging (RMSE={rmse_ok:.2f})')
plt.colorbar(im1, ax=axes[1])

# Kriging variance
im2 = axes[2].pcolormesh(gx, gy, v_ok.reshape(ny, nx),
                          cmap='YlOrRd', shading='auto')
axes[2].scatter(ch[:, 0], ch[:, 1], c='cyan', marker='v', s=20, zorder=5)
axes[2].set_title('Kriging Variance')
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_aspect('equal')
fig.suptitle('PyBME: Kriging Tutorial', fontsize=13)
fig.tight_layout()
plt.show()